# Layer test: Sentinel-2 (RGB / NIR / SWIR)

Ingest and display the **Sentinel-2** input layer. QA60 cloud mask, `CLOUDY_PIXEL_PERCENTAGE < 10`, and median composite match `761_Lab2.ipynb` / `lab2_helpers.py`.

SWIR bands **B11** and **B12** are kept because the project input is RGB + NIR + SWIR. NDWI is McFeeters `normalizedDifference(['B3', 'B8'])` from the lab helper.


In [ ]:
import sys
from pathlib import Path

import ee
import geemap

_root = Path.cwd()
_nb = _root / "notebooks" if (_root / "notebooks" / "layer_config.py").exists() else _root
sys.path.insert(0, str(_nb))
import layer_config as cfg

ee.Initialize(project=cfg.GEE_PROJECT)

aoi = ee.Geometry.Rectangle(cfg.AOI_BOUNDS)
print("GEE project:", cfg.GEE_PROJECT)
print("AOI:", cfg.AOI_BOUNDS)
print("Dates:", cfg.START_DATE, "→", cfg.END_DATE)

In [ ]:
def mask_s2_clouds(image):
    qa = image.select("QA60")
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
    return image.updateMask(mask).divide(10000).select(["B2", "B3", "B4", "B8", "B11", "B12"])

s2_col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(cfg.START_DATE, cfg.END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
)

n = s2_col.size().getInfo()
print("S2 scenes after date / cloud metadata filter:", n)
assert n > 0, "No Sentinel-2 scenes. Widen dates or raise CLOUDY_PIXEL_PERCENTAGE in this cell."

info = s2_col.first().getInfo()
print("First scene id:", info["id"])
print("Scene cloud %:", info["properties"].get("CLOUDY_PIXEL_PERCENTAGE"))

In [ ]:
s2 = s2_col.map(mask_s2_clouds).median().clip(aoi)
ndwi = s2.normalizedDifference(["B3", "B8"]).rename("NDWI")

rgb = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}
swir = {"bands": ["B12", "B8", "B4"], "min": 0, "max": 0.3}
ndwi_vis = {"min": -0.5, "max": 0.5, "palette": ["brown", "white", "blue"]}

Map = geemap.Map(center=cfg.MAP_CENTER, zoom=cfg.MAP_ZOOM, basemap="HYBRID")
Map.addLayer(s2, rgb, "S2 RGB")
Map.addLayer(s2, swir, "S2 SWIR-NIR-Red")
Map.addLayer(ndwi, ndwi_vis, "NDWI")
Map.addLayer(aoi, {"color": "red"}, "AOI")
Map